In [1]:
import os
import base64
from dotenv import load_dotenv
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad


# CONFIGURATION
ENV_FILE = "/home/yogavarman/Projects/Config/config.env"

# LOAD ENV FILE
load_dotenv(ENV_FILE,override=True)

# DECRYPT CONFIGURATION
def decrypt_config(server_name: str) -> dict:
    # Get APP_KEY
    app_key = os.getenv("APP_KEY")
    if not app_key:
        raise ValueError(
            "APP_KEY not found in config.env"
        )

    # Get encrypted server value
    encrypted = os.getenv(server_name)
    if not encrypted:
        raise ValueError(
            f"{server_name} not found in config.env"
        )

    # Decode APP_KEY
    try:
        key = base64.b64decode(app_key)

    except Exception as e:
        raise ValueError(
            f"Invalid APP_KEY: {e}"
        )

    # Validate AES key
    if len(key) not in (16, 24, 32):
        raise ValueError(
            f"Invalid AES key length: {len(key)} bytes"
        )
    
    # Decode encrypted value
    try:
        encrypted_data = base64.b64decode(
            encrypted
        )
    except Exception as e:
        raise ValueError(
            f"Invalid encrypted data: {e}"
        )

    # Validate IV + ciphertext
    if len(encrypted_data) <= AES.block_size:
        raise ValueError(
            f"Invalid encrypted data for {server_name}"
        )

    # Extract IV
    iv = encrypted_data[:AES.block_size]

    # Extract ciphertext
    ciphertext = encrypted_data[AES.block_size:]

    # Ciphertext must be multiple of AES block size
    if len(ciphertext) % AES.block_size != 0:
        raise ValueError(
            f"Invalid ciphertext length for {server_name}"
        )
    
    # AES CBC
    cipher = AES.new(key,AES.MODE_CBC,iv)

    # Decrypt
    try:
        decrypted = cipher.decrypt(ciphertext)
        decrypted = unpad(decrypted,AES.block_size)
        decrypted = decrypted.decode("utf-8")
    except ValueError:
        raise ValueError(
            f"Unable to decrypt {server_name}. "
            "APP_KEY and encrypted value do not match."
        )
    
    # Convert to dictionary
    config = {}
    for line in decrypted.splitlines():
        line = line.strip()
        if not line:
            continue
        if "=" in line:
            name, value = line.split("=",1)
            config[name.strip()] = value.strip()
    return config


# TEST DB SERVER
db_server = decrypt_config("DB_SERVER")

print()
print("DB SERVER")
print(db_server)




DB SERVER
{'HOST': '172.23.160.1', 'PORT': '5432', 'DATABASE': 'postgres', 'USERNAME': 'postgres', 'PASSWORD': 'admin123'}


In [ ]:
from datetime import datetime
import sys
sys.path.append("/home/yogavarman/Projects/FoodChain")

from Config.db import JDBC_URL, DB_PROPERTIES, DATABASE_URL,get_conn
conn = get_conn()
def generate_batch_reference(cur):

    count_date = datetime.now().strftime("%Y%m%d")

    cur.execute("""
        INSERT INTO foodchain.idgen (
            count_date,
            req_count
        )
        VALUES (%s, 1)

        ON CONFLICT (count_date)
        DO UPDATE
        SET req_count = foodchain.idgen.req_count + 1

        RETURNING req_count
    """, (count_date,))

    result = cur.fetchone()

    seq = result[0] if result else 1

    return f"{count_date}{seq:02d}"


NameError: name 'cur' is not defined

In [4]:
conn = get_conn()
cur = conn.cursor()

batch_reference_id = generate_batch_reference(cur)

conn.commit()

print("Batch Reference ID:", batch_reference_id)

Batch Reference ID: 2026082001


In [14]:
def send_user_credentials(email, first_name, username, password):
    import smtplib
    from email.message import EmailMessage

    SMTP_HOST = "smtp.gmail.com"
    SMTP_PORT = 587

    SMTP_USERNAME = "myworkasde@gmail.com"
    SMTP_PASSWORD = "afcz mocy xhxj uqli"

    message = EmailMessage()

    message["Subject"] = "FoodChain Account Created"
    message["From"] = SMTP_USERNAME
    message["To"] = email

    message.set_content(
        f"""Hello {first_name},

Your FoodChain account has been created successfully.

Username: {username}
Temporary Password: {password}

Please login to FoodChain using these credentials.

Regards,
FoodChain Team
"""
    )

    with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
        server.ehlo()
        server.starttls()
        server.ehlo()

        server.login(
            SMTP_USERNAME,
            SMTP_PASSWORD
        )

        server.send_message(message)

    print(f"Email sent successfully to {email}")
    
    
email = "akyogavarman39@gmail.com"
first_name = "YOGAVARMAN"
username = 2021
temporary_password = "hfhsdkkwmdnfd"

send_user_credentials(
    email=email.strip(),
    first_name=first_name.strip(),
    username=username,
    password=temporary_password
)

Email sent successfully to akyogavarman39@gmail.com
